# Figure S12: Within- and Cross-Region Multivariate Theta-HFA Phase-Amplitude Coupling (GC-PAC)

| Field | Value |
|---|---|
| **Manuscript panels / stats** | Fig S12A (example HPC theta phase -> ACC HFA amplitude comodulogram during recognition cue for positive vs neutral RPE), Fig S12B (mean z-scored multivariate PAC across RPE classes for HPC-ACC, ACC-ACC, and HPC-HPC; 250 block-swap surrogates; mixed-effects regression all ps > 0.05) |
| **Inputs** | `$MEMORYBANDIT_DATA_DIR/projects/guLab/Salman/EphysAnalyses/{subj}/neural/Day1/cue_on-epo.fif`, `Day1_reref_elec_df`, `learn_df_RWH_v2.csv`, `full_df_RWH_v2.csv` |
| **Outputs** | `all_gcpac_subject_dual_event.csv`, `multitarget_mixedlm_fdr.csv`, `figures/MemoryBandit-MV_GCPAC_HPC_ACC_RPE/*` |
| **Dependencies** | `membandit`, patched `tensorpac` (`github.com/seqasim/tensorpac` @ `33dcbf5`; set `TENSORPAC_ROOT`), `statsmodels`, R `lme4` (`R/lmer_neural.R`) |
| **Expected runtime** | ~1-2 hours from raw epochs (250 surrogates/trial); seconds when reading per-subject `.npz` caches |
| **Upstream steps** | `01_behavior/Fig1DEG_S1_TableS1_modeling_EMU.ipynb`; run `Rscript R/lmer_neural.R` for R `lmer` fits |


# Multivariate Gaussian-copula PAC by RPE category (HPC phase → ACC amplitude)

Trial-level multivariate GC-PAC (Tensorpac) between ROI-hemisphere channel groups:

- Theta phase (4–8 Hz) and HFA amplitude (70–200 Hz), wavelet filtering, 1000 ms edge trim
- Copula-normalized multivariate mutual information per trial and frequency-bin pair
- Surrogate z-scoring with tensorpac block-swap null (`idpac=(6, 2, 0)`)
- Trial alignment via `build_rpe_trial_map`; inference via mixed models and FDR (aligned with CaCoh/PSI notebooks)


## 0. Setup


In [ ]:
# membandit setup
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

import mne

from membandit.plotting_style import set_plot_style
from membandit.behavior import load_behavior_tables, preprocess_learn_df, preprocess_combined_df
from membandit.io import load_subject_epochs
from membandit.gcpac import (
    GcpacParams,
    alignment_valid_mask,
    ensure_tensorpac,
    load_elec_df,
    make_pac_object,
    process_subject_gcpac_unit,
    try_load_subject_gcpac_cache,
)
from membandit.trial_connectivity import run_trial_stats_and_plots
from membandit.subject_covariates import (
    load_acc_reinstatement_table,
    load_hpc_memory_classification_table,
    load_acc_hpc_hit_rsa_table,
    correlate_subject_metric,
)

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")
set_plot_style()
ensure_tensorpac()  # must run before import tensorpac (loads patched checkout)
import tensorpac
print("tensorpac", getattr(tensorpac, "__version__", "?"), "from", tensorpac.__file__)


## 1. Config


In [ ]:
EVENT = "cue_on"  # headline event: "feedback_start" or "cue_on"

EVENT_SPECS = {
    "cue_on": dict(tmin=0.0, tmax=1.0, behavior="combined_df"),
}
if EVENT not in EVENT_SPECS:
    raise ValueError(f"EVENT must be one of {list(EVENT_SPECS)}")

SPEC = EVENT_SPECS[EVENT]
TMIN = float(SPEC["tmin"])
TMAX = float(SPEC["tmax"])
BEHAVIOR_SOURCE = SPEC["behavior"]

DAY = "Day1"
PHASE_ROI = "HPC"
AMP_ROI = "ACC"
ROI_PAIRS = [("HPC", "ACC"), ("ACC", "HPC"), ("HPC", "HPC"), ("ACC", "ACC")]
HEMIS = ("l", "r")
TARGET_SFREQ = 1000.0  # no downsampling (only resample when target < native sfreq)
MIN_TRIALS = 5
SEED = 42
RPE_CATS = ("negative", "neutral", "positive")
COLORS = {"negative": "#d62728", "neutral": "#7f7f7f", "positive": "#2ca02c"}
OVERWRITE_CACHE = False
GCPAC_SMOKE = os.environ.get("GCPAC_SMOKE", "").strip().lower() in ("1", "true", "yes")

GCPAC_PARAMS = GcpacParams(random_state=SEED, n_jobs=4 if not GCPAC_SMOKE else 1)
BAND_TAG = GCPAC_PARAMS.band_tag
PARAMS_HASH = GCPAC_PARAMS.params_hash()

base_dir = paths.get_base_dir()
load_dir = f"{base_dir}/work/qasims01/MemoryBanditData/EMU"
savepath = os.path.join(
    base_dir,
    "projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/mv_gcpac_rpe",
    EVENT,
    BAND_TAG,
)
plot_savepath = os.path.join(
    os.environ.get(
        "MEMORYBANDIT_PLOT_DIR",
        paths.figure_output_root(),
    ),
    "MemoryBandit-MV_GCPAC_HPC_ACC_RPE",
    BAND_TAG,
)
try:
    os.makedirs(plot_savepath, exist_ok=True)
except OSError:
    plot_savepath = os.path.join(str(Path.cwd()), "figures", "MemoryBandit-MV_GCPAC_HPC_ACC_RPE", BAND_TAG)
    os.makedirs(plot_savepath, exist_ok=True)
os.makedirs(savepath, exist_ok=True)

PAC_OBJ = make_pac_object(GCPAC_PARAMS)
print(f"EVENT={EVENT} window={TMIN}–{TMAX}s behavior={BEHAVIOR_SOURCE}")
print(f"params_hash={PARAMS_HASH} band={BAND_TAG}")
print(f"cache: {savepath}")
print(f"plots: {plot_savepath}")


## 2. Subjects and behavior


In [ ]:
demo_file = f"{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx"
demo_data = pd.read_excel(demo_file, engine="openpyxl")
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df["MSSMCode"].isin(["MS015", "MS024", "MS034", "MS038"])]
subjects = sorted(subj_df["MSSMCode"].dropna().unique().tolist())
if GCPAC_SMOKE and subjects:
    subjects = subjects[:1]

learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)
beh_df = learn_df if BEHAVIOR_SOURCE == "learn_df" else combined_df

print("Subjects:", len(subjects))
print("behavior source:", BEHAVIOR_SOURCE, beh_df.shape)


## 3. GCPAC helpers + per-subject cache


In [ ]:
def process_subject_headline(subj_id):
    # Load full epochs (no crop): tensorpac buf_ms edge trim needs padding around the
    # analysis window. Pre-cropping to TMIN/TMAX leaves zero samples after trim.
    cache_dir = savepath
    trial_parts = []
    meta = {"subject": subj_id, "hemi": [], "n_phase": [], "n_amp": []}
    epoch_data = None
    elec_df = None
    for hemi in HEMIS:
        cached = try_load_subject_gcpac_cache(
            cache_dir=cache_dir,
            subject=subj_id,
            event=EVENT,
            phase_roi=PHASE_ROI,
            amp_roi=AMP_ROI,
            hemi=hemi,
            params=GCPAC_PARAMS,
            behavior_df=beh_df,
            overwrite_cache=OVERWRITE_CACHE,
            day=DAY,
        )
        if cached is not None:
            rec = cached
        else:
            if epoch_data is None:
                epoch_data = load_subject_epochs(
                    subj_id=subj_id,
                    base_dir=base_dir,
                    event=EVENT,
                    roi_source=PHASE_ROI,
                    roi_target=AMP_ROI,
                    tmin=None,
                    tmax=None,
                    target_sfreq=TARGET_SFREQ,
                    day=DAY,
                )
                if epoch_data is None:
                    return None
                elec_df = load_elec_df(base_dir, subj_id, day=DAY)
                epoch_data = dict(epoch_data)
                epoch_data["epochs"] = epoch_data["epochs"].copy()
            rec = process_subject_gcpac_unit(
                subject=subj_id,
                epoch_data=epoch_data,
                elec_df=elec_df,
                behavior_df=beh_df,
                event=EVENT,
                phase_roi=PHASE_ROI,
                amp_roi=AMP_ROI,
                hemi=hemi,
                params=GCPAC_PARAMS,
                pac_obj=PAC_OBJ,
                cache_dir=cache_dir,
                overwrite_cache=OVERWRITE_CACHE,
                min_trials_per_category=MIN_TRIALS,
                day=DAY,
            )
        if rec is None:
            continue
        tdf = rec["trial_df"].copy()
        tdf = tdf[alignment_valid_mask(tdf)].dropna(subset=["rpe_cat", "trial_pac_z"])
        if len(tdf) == 0:
            continue
        trial_parts.append(tdf)
        meta["hemi"].append(hemi)
        meta["n_phase"].append(len(rec["phase_channels"]))
        meta["n_amp"].append(len(rec["amp_channels"]))

    if not trial_parts:
        return None
    out_df = pd.concat(trial_parts, ignore_index=True)
    return {"trial_df": out_df, "meta": meta}


def process_subject_pair_hemi(subj_id, event, phase_roi, amp_roi, hemi, behavior_df, tmin, tmax):
    del tmin, tmax  # full epoch load; event window used for reporting only
    cache_dir = os.path.join(
        base_dir,
        "projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/mv_gcpac_rpe",
        event,
        BAND_TAG,
    )
    cached = try_load_subject_gcpac_cache(
        cache_dir=cache_dir,
        subject=subj_id,
        event=event,
        phase_roi=phase_roi,
        amp_roi=amp_roi,
        hemi=hemi,
        params=GCPAC_PARAMS,
        behavior_df=behavior_df,
        overwrite_cache=OVERWRITE_CACHE,
        day=DAY,
    )
    if cached is not None:
        return cached

    epoch_data = load_subject_epochs(
        subj_id=subj_id,
        base_dir=base_dir,
        event=event,
        roi_source=phase_roi,
        roi_target=amp_roi,
        tmin=None,
        tmax=None,
        target_sfreq=TARGET_SFREQ,
        day=DAY,
    )
    if epoch_data is None:
        return None
    elec_df = load_elec_df(base_dir, subj_id, day=DAY)
    epoch_data = dict(epoch_data)
    epoch_data["epochs"] = epoch_data["epochs"].copy()
    return process_subject_gcpac_unit(
        subject=subj_id,
        epoch_data=epoch_data,
        elec_df=elec_df,
        behavior_df=behavior_df,
        event=event,
        phase_roi=phase_roi,
        amp_roi=amp_roi,
        hemi=hemi,
        params=GCPAC_PARAMS,
        pac_obj=PAC_OBJ,
        cache_dir=cache_dir,
        overwrite_cache=OVERWRITE_CACHE,
        min_trials_per_category=MIN_TRIALS,
        day=DAY,
    )


## 4. Main loop, group aggregate, and headline plots/stats


In [ ]:
results = {}
t0 = time.time()
for subj_id in tqdm(subjects, desc=f"MV-GCPAC {PHASE_ROI}->{AMP_ROI}"):
    rec = process_subject_headline(subj_id)
    if rec is not None:
        results[subj_id] = rec
print(f"Finished {len(results)} subjects in {(time.time() - t0) / 60:.1f} min")

trial_df = pd.concat([r["trial_df"] for r in results.values()], ignore_index=True)
trial_df["rpe_cat"] = pd.Categorical(trial_df["rpe_cat"], categories=list(RPE_CATS), ordered=False)
print("trial rows:", len(trial_df), "subjects:", trial_df["subject"].nunique())

keep_subjects = sorted(trial_df["subject"].unique())
print("subjects with data:", len(keep_subjects))

subj_means = (
    trial_df.groupby(["subject", "rpe_cat"], observed=True)["trial_pac_z"]
    .mean()
    .unstack("rpe_cat")
    .reindex(columns=list(RPE_CATS))
)

print("\nOne-sample Wilcoxon vs 0 (subject means)")
for cat in RPE_CATS:
    stat = wilcoxon(subj_means[cat].dropna(), zero_method="wilcox", alternative="two-sided")
    print(f"{cat}: W={stat.statistic:.3f}, p={stat.pvalue:.5f}")

pair_keys = [("negative", "positive"), ("negative", "neutral"), ("positive", "neutral")]
print("\nPaired Wilcoxon contrasts")
for a, b in pair_keys:
    sub = subj_means[[a, b]].dropna()
    if len(sub) < 2:
        print(f"{a} - {b}: insufficient N")
        continue
    stat = wilcoxon(sub[a], sub[b], zero_method="wilcox", alternative="two-sided")
    print(f"{a} - {b}: W={stat.statistic:.3f}, p={stat.pvalue:.5f}")

fig, ax = plt.subplots(1, 1, figsize=(5.2, 4.2))
x = np.arange(len(RPE_CATS), dtype=float)
for _, row in subj_means.iterrows():
    ax.plot(x, row.to_numpy(float), color="0.78", lw=0.9, alpha=0.9)
means = subj_means.mean(axis=0)
sems = subj_means.std(axis=0, ddof=1) / np.sqrt(max(1, len(subj_means)))
for i, cat in enumerate(RPE_CATS):
    ax.errorbar(
        x[i], means[cat], yerr=sems[cat], fmt="o", ms=8, capsize=3,
        color=COLORS[cat], label=cat, zorder=5,
    )
ax.axhline(0.0, color="k", ls="--", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(list(RPE_CATS))
ax.set_ylabel("Multivariate PAC z (trial mean)")
ax.set_title(f"{EVENT} {PHASE_ROI}->{AMP_ROI} · N={len(subj_means)}")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
fig.tight_layout()
fig_path = os.path.join(plot_savepath, f"gcpac_{PHASE_ROI}_{AMP_ROI}_{EVENT}.png")
fig.savefig(fig_path, dpi=300, bbox_inches="tight")
print("saved figure", fig_path)
plt.show()

csv_path = os.path.join(savepath, f"gcpac_trial_{EVENT}_{PHASE_ROI}_{AMP_ROI}.csv")
trial_df.to_csv(csv_path, index=False)
print("saved", csv_path)


## 5. Pair-level mixed models (trial_connectivity)


In [ ]:
if len(trial_df) == 0:
    raise RuntimeError("No trial rows for MixedLM")

bundle = run_trial_stats_and_plots(
    df=trial_df,
    z_col="trial_pac_z",
    metric_name="mv_gcpac",
    event=EVENT,
    save_dir=savepath,
    plot_dir=plot_savepath,
    categories=RPE_CATS,
    min_n=MIN_TRIALS,
    pair_vc=True,
    colors=COLORS,
    ylabel="Multivariate PAC z",
    title_prefix=f"{PHASE_ROI}->{AMP_ROI}",
)
print(bundle.diagnostics)
print(bundle.fixed_effects)
print(bundle.pairwise_subject)


## 6. Multi-target dual-event pipeline


In [ ]:
EVENT_SPECS_MULTI = {
    "cue_on": dict(tmin=0.0, tmax=1.0, behavior_df=combined_df),
}
# Aggregate CSV lives under band tag root (not under headline EVENT), so re-runs
# with EVENT=cue_on still find prior dual-event exports.
savepath_multi = os.path.join(
    base_dir,
    "projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/mv_gcpac_rpe",
    "multi_target_dual_event",
    BAND_TAG,
)
os.makedirs(savepath_multi, exist_ok=True)
all_gcpac_csv = os.path.join(savepath_multi, "all_gcpac_subject_dual_event.csv")
attrition_csv = os.path.join(savepath_multi, "attrition_dual_event.csv")

if os.path.exists(all_gcpac_csv) and not OVERWRITE_CACHE:
    all_gcpac_df = pd.read_csv(all_gcpac_csv)
    print("loaded aggregate CSV", all_gcpac_csv, "rows:", len(all_gcpac_df))
else:
    rows = []
    attrition = []
    n_cache_hit = 0
    n_compute = 0
    for event, spec in EVENT_SPECS_MULTI.items():
        bdf = spec["behavior_df"]
        for phase_roi, amp_roi in ROI_PAIRS:
            for hemi in HEMIS:
                for subj_id in tqdm(subjects, desc=f"GCPAC {event} {phase_roi}->{amp_roi} {hemi}"):
                    rec = process_subject_pair_hemi(
                        subj_id, event, phase_roi, amp_roi, hemi, bdf, spec["tmin"], spec["tmax"]
                    )
                    if rec is None:
                        attrition.append(
                            {
                                "subject": subj_id,
                                "event": event,
                                "phase_roi": phase_roi,
                                "amp_roi": amp_roi,
                                "hemi": hemi,
                                "status": "skip",
                            }
                        )
                        continue
                    if rec.get("from_cache"):
                        n_cache_hit += 1
                    else:
                        n_compute += 1
                    tdf = rec["trial_df"]
                    tdf = tdf[alignment_valid_mask(tdf)].dropna(subset=["rpe_cat", "trial_pac_z"])
                    for _, row in tdf.iterrows():
                        rows.append(
                            {
                                "subject": row["subject"],
                                "event": row["event"],
                                "phase_roi": row["phase_roi"],
                                "amp_roi": row["amp_roi"],
                                "hemi": row["hemi"],
                                "pair_label": row["pair_label"],
                                "epoch_idx": int(row["epoch_idx"]),
                                "rpe_cat": row["rpe_cat"],
                                "trial_pac_z": float(row["trial_pac_z"]),
                            }
                        )
                    attrition.append(
                        {
                            "subject": subj_id,
                            "event": event,
                            "phase_roi": phase_roi,
                            "amp_roi": amp_roi,
                            "hemi": hemi,
                            "status": "ok",
                            "from_cache": bool(rec.get("from_cache", False)),
                            "n_phase": len(rec["phase_channels"]),
                            "n_amp": len(rec["amp_channels"]),
                        }
                    )
    all_gcpac_df = pd.DataFrame(rows)
    all_gcpac_df.to_csv(all_gcpac_csv, index=False)
    pd.DataFrame(attrition).to_csv(attrition_csv, index=False)
    print("saved", all_gcpac_csv)
    print("saved", attrition_csv)
    print(f"subject-unit cache hits={n_cache_hit} recomputes={n_compute}")

print("rows:", len(all_gcpac_df))


## 7. MVGC-style bar + strip plot (cue_on, HPC→ACC)


In [ ]:
plot_event = "cue_on"
pairs = [("HPC", "ACC"), ("ACC", "ACC"), ("HPC", "HPC")]
cats = list(RPE_CATS)

fig, axes = plt.subplots(1, 3, figsize=(14.5, 5.0), sharey=True)

for ax, (plot_phase, plot_amp) in zip(axes, pairs):
    plot_df = all_gcpac_df[
        all_gcpac_df["event"].eq(plot_event)
        & all_gcpac_df["phase_roi"].eq(plot_phase)
        & all_gcpac_df["amp_roi"].eq(plot_amp)
        & all_gcpac_df["rpe_cat"].isin(cats)
    ].copy()

    if len(plot_df) == 0:
        ax.set_title(f"{plot_phase}-{plot_amp}")
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        continue

    # Match old figure: within-subject z of trial PAC, then subject means by RPE
    plot_df["pac_z_within"] = plot_df.groupby(
        ["subject", "phase_roi", "amp_roi", "event"], observed=True
    )["trial_pac_z"].transform(
        lambda x: (x - x.mean()) / (x.std(ddof=0) if x.std(ddof=0) > 0 else np.nan)
    )
    if not plot_df["pac_z_within"].notna().any():
        plot_df["pac_z_within"] = plot_df["trial_pac_z"]

    subj_means_plot = (
        plot_df.groupby(["subject", "rpe_cat"], observed=True)["pac_z_within"]
        .mean()
        .reset_index()
    )
    bar_df = (
        subj_means_plot.groupby("rpe_cat", observed=True)["pac_z_within"]
        .agg(["mean", "sem"])
        .reindex(cats)
        .reset_index()
    )

    x = np.arange(len(cats))
    ax.bar(
        x,
        bar_df["mean"],
        yerr=bar_df["sem"],
        width=0.62,
        color=[COLORS[c] for c in cats],
        alpha=0.65,
        capsize=0,
        zorder=2,
    )
    sns.stripplot(
        data=subj_means_plot,
        x="rpe_cat",
        y="pac_z_within",
        order=cats,
        hue="rpe_cat",
        palette=COLORS,
        dodge=False,
        jitter=0.18,
        alpha=0.55,
        edgecolor="black",
        linewidth=0.8,
        size=4.0,
        ax=ax,
        zorder=4,
        legend=False,
    )
    ax.axhline(0, color="black", linewidth=1.2, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(cats)
    ax.set_xlabel("RPE (class)")
    ax.set_title(f"{plot_phase}-{plot_amp}")
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

axes[0].set_ylabel("multivariate PAC (z)")
axes[0].set_ylim(-0.43, 0.49)
axes[0].set_yticks([-0.4, 0.0, 0.4])

fig.tight_layout()
out_pdf = os.path.join(plot_savepath, f"gcpac_mixed_strip_{plot_event}_3pairs.pdf")
fig.savefig(out_pdf, dpi=300, bbox_inches="tight")
print("saved", out_pdf)
plt.show()


## 8. Per-pair mixed models + FDR (multi-target)


In [ ]:
mm_rows = []
for (event, phase_roi, amp_roi, hemi), sub in all_gcpac_df.groupby(
    ["event", "phase_roi", "amp_roi", "hemi"], observed=True
):
    sub = sub.dropna(subset=["trial_pac_z", "rpe_cat"]).copy()
    if sub["subject"].nunique() < 2:
        continue
    counts = sub.groupby(["subject", "rpe_cat"], observed=True).size().unstack(fill_value=0)
    counts = counts.reindex(columns=list(RPE_CATS), fill_value=0)
    complete = counts.index[(counts[list(RPE_CATS)] >= MIN_TRIALS).all(axis=1)]
    sub = sub[sub["subject"].isin(complete)]
    if len(sub) < MIN_TRIALS * 3:
        continue
    sub["rpe_cat"] = pd.Categorical(sub["rpe_cat"], categories=list(RPE_CATS), ordered=False)
    try:
        mfit = smf.mixedlm(
            "trial_pac_z ~ C(rpe_cat, Treatment('positive'))",
            data=sub,
            groups=sub["subject"],
            re_formula="1",
        ).fit(method="lbfgs", maxiter=500, reml=False)
    except Exception:
        continue
    for term, pval in mfit.pvalues.items():
        if "rpe_cat" not in str(term):
            continue
        mm_rows.append(
            {
                "event": event,
                "phase_roi": phase_roi,
                "amp_roi": amp_roi,
                "hemi": hemi,
                "term": str(term),
                "p_value": float(pval),
                "n_subjects": int(sub["subject"].nunique()),
                "n_rows": int(len(sub)),
            }
        )

mm_df = pd.DataFrame(mm_rows)
if len(mm_df):
    _, qvals, _, _ = multipletests(mm_df["p_value"].to_numpy(float), method="fdr_bh")
    mm_df["p_fdr"] = qvals
    mm_out = os.path.join(savepath_multi, "multitarget_mixedlm_fdr.csv")
    mm_df.to_csv(mm_out, index=False)
    print("saved", mm_out)
    print(mm_df.sort_values("p_fdr").head(20).to_string(index=False))
else:
    print("No mixed models converged")


## 9. Subject-level covariate correlations


In [ ]:
gcpac_subject = (
    trial_df.groupby(["subject", "rpe_cat"], observed=True)["trial_pac_z"]
    .mean()
    .reset_index()
    .rename(columns={"trial_pac_z": "gcpac_z"})
)
gcpac_subject = gcpac_subject.rename(columns={"subject": "Subject"})
print("subject x rpe rows:", len(gcpac_subject))

reinstatement_tbl = load_acc_reinstatement_table(base_dir=base_dir)
rsa_subject_tbl = load_acc_hpc_hit_rsa_table(base_dir=base_dir, n_jobs=4)
hpc_mem_tbl = load_hpc_memory_classification_table(base_dir=base_dir)

correlate_subject_metric(
    x_tbl=gcpac_subject,
    metric_tbl=reinstatement_tbl,
    x_col="gcpac_z",
    x_label=f"{EVENT} {PHASE_ROI}->{AMP_ROI} MV-GCPAC z",
    y_col="ACC_MeanAccuracy",
    y_label="ACC RPE reactivation (%)",
    rpe_cats=RPE_CATS,
    plot_savepath=plot_savepath,
    fig_stem=f"gcpac_{EVENT}_vs_ACC_reinstatement",
    title=f"ACC RPE reactivation vs {EVENT} MV-GCPAC",
)
correlate_subject_metric(
    x_tbl=gcpac_subject,
    metric_tbl=rsa_subject_tbl,
    x_col="gcpac_z",
    x_label=f"{EVENT} {PHASE_ROI}->{AMP_ROI} MV-GCPAC z",
    y_col="ACC_HPC_HitRSA_cluster_mean",
    y_label="ACC–HPC hit RSA (cluster mean)",
    rpe_cats=RPE_CATS,
    plot_savepath=plot_savepath,
    fig_stem=f"gcpac_{EVENT}_vs_ACC_HPC_hitRSA",
    title=f"ACC–HPC hit RSA vs {EVENT} MV-GCPAC",
)
correlate_subject_metric(
    x_tbl=gcpac_subject,
    metric_tbl=hpc_mem_tbl,
    x_col="gcpac_z",
    x_label=f"{EVENT} {PHASE_ROI}->{AMP_ROI} MV-GCPAC z",
    y_col="HPC_MemoryClassAccuracy",
    y_label="HPC memory classification accuracy",
    rpe_cats=RPE_CATS,
    plot_savepath=plot_savepath,
    fig_stem=f"gcpac_{EVENT}_vs_HPC_memory_classification",
    title=f"HPC memory classification vs {EVENT} MV-GCPAC",
)
